In [ ]:


# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# IMPORTAR BIBLIOTECAS ---
import sys
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# CAMINHOS ---
SCRIPT_DIR = Path(__file__).resolve().parent
ANALYTICS_DIR = SCRIPT_DIR.parent
PROJECT_ROOT = Path(__file__).resolve().parents[3]

# SPARK ---
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# PREPARAÇÃO PARA ANALISAR GOLD 02 - Quais perfis profissionais são mais valorizados pelo mercado?
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CAMINHO DO ARQUIVO
caminho_gold_02 = (PROJECT_ROOT/ "Gold"/ "perguntas_negocio"/ "gold_02_perfis_valorizados")

# BUSCAR CSVs GERADOS PELO SPARK ---
arquivos_gold_02 = [
    str(arquivo)
    for arquivo in caminho_gold_02.glob("part-*.csv")
]
print("\nARQUIVOS ENCONTRADOS:")
print(arquivos_gold_02)

if not arquivos_gold_02:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {caminho_gold_02}"
    )

# CARREGAR GOLD 02 ---
df_perfis = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_02)
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------



# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# INSPEÇÃO INICIAL DA GOLD 02 - PARA IDENTIFICAR O QUE EXISTE
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("1. INSPEÇÃO INICIAL DA GOLD 02")
print("=" * 100)

# AMOSTRA ---
df_perfis.show(30,truncate=False)

# SCHEMA ---
df_perfis.printSchema()

# TAMANHO DA BASE ---
print("Quantidade de linhas:",df_perfis.count())
print("Colunas:",df_perfis.columns)

# VARIÁVEIS EXISTENTES ---
print("\nVARIÁVEIS EXISTENTES:")
(df_perfis
    .select("variavel")
    .distinct()
    .orderBy("variavel")
    .show(
        100,
        truncate=False
    )
)

# EDIÇÕES EXISTENTES ---
print("\nEDIÇÕES EXISTENTES:")
(df_perfis
    .select("edicao")
    .distinct()
    .orderBy("edicao")
    .show(
        20,
        truncate=False
    )
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A Gold 02 possui 986 registros das três edições analisadas.
A base contém exclusivamente a variável de faixa salarial, segmentada por cargo e nível.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------



# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# INSPEÇÃO DAS TAXONOMIAS
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("2. INSPEÇÃO DAS TAXONOMIAS")
print("=" * 100)


# FAIXAS SALARIAIS POR EDIÇÃO ---
print("\nFAIXAS SALARIAIS POR EDIÇÃO:")
(df_perfis
    .select("edicao","valor")
    .distinct()
    .orderBy("edicao","valor")
    .show(100,truncate=False)
)

# QUANTIDADE DE FAIXAS SALARIAIS POR EDIÇÃO ---
print("\nQUANTIDADE DE FAIXAS SALARIAIS POR EDIÇÃO:")
(df_perfis
    .groupBy("edicao")
    .agg(F.countDistinct("valor").alias("qtd_faixas_salariais"))
    .orderBy("edicao")
    .show(truncate=False)
)

# NÍVEIS POR EDIÇÃO ---
print("\nNÍVEIS POR EDIÇÃO:")
(df_perfis
    .select("edicao","nivel")
    .distinct()
    .orderBy("edicao","nivel")
    .show(100,truncate=False)
)

# QUANTIDADE DE CARGOS POR EDIÇÃO ---
print("\nQUANTIDADE DE CARGOS POR EDIÇÃO:")
(df_perfis
    .groupBy("edicao")
    .agg(F.countDistinct("cargo_atual").alias("qtd_cargos"))
    .orderBy("edicao")
    .show(truncate=False)
)

# CARGOS POR EDIÇÃO ---
print("\nCARGOS POR EDIÇÃO:")
(df_perfis
    .select("edicao","cargo_atual")
    .distinct()
    .orderBy("edicao","cargo_atual")
    .show(200,truncate=False)
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
As taxonomias não são totalmente iguais entre as edições:
- 2023-2024 possui 14 faixas salariais, enquanto as demais possuem 13.
- Especialista/Staff+ aparece apenas em 2025-2026.
- A quantidade de cargos varia entre 17 em 2023-2024 e 15 nas edições seguintes.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------



# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# VALIDAÇÃO DO DENOMINADOR
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("3. VALIDAÇÃO DO DENOMINADOR")
print("=" * 100)


validacao_denominador = (df_perfis
    .groupBy("edicao","cargo_atual","nivel")
    .agg(
        F.sum("contagem").alias("soma_contagem"),
        F.max("total_respondentes").alias("total_respondentes"),
        F.countDistinct("total_respondentes").alias("qtd_totais_distintos"),
        F.round(F.sum("pct_na_dimensao"),2).alias("soma_percentual"))
    .withColumn("contagem_bate_total",F.col("soma_contagem")== F.col("total_respondentes"))
    .orderBy("edicao","cargo_atual","nivel")
)
validacao_denominador.show(200,truncate=False)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
O denominador está consistente: em todos os grupos, a soma das contagens corresponde ao total de respondentes.
Pequenas diferenças na soma dos percentuais decorrem de arredondamento.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------



# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# COMPARABILIDADE ENTRE AS EDIÇÕES
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("4. COMPARABILIDADE ENTRE AS EDIÇÕES")
print("=" * 100)

# FAIXAS SALARIAIS ---
print("\nCOMPARABILIDADE DAS FAIXAS SALARIAIS:")

comparabilidade_faixas = (df_perfis
    .select("edicao","valor")
    .distinct()
    .groupBy("valor")
    .agg(F.countDistinct("edicao").alias("qtd_edicoes"))
    .orderBy("qtd_edicoes","valor")
)
comparabilidade_faixas.show(100,truncate=False)

# NÍVEIS ---
print("\nCOMPARABILIDADE DOS NÍVEIS:")

comparabilidade_niveis = (df_perfis
    .select("edicao","nivel")
    .distinct()
    .groupBy("nivel")
    .agg(F.countDistinct("edicao").alias("qtd_edicoes"))
    .orderBy("qtd_edicoes","nivel")
)
comparabilidade_niveis.show(100,truncate=False)

# CARGOS ---
print("\nCOMPARABILIDADE DOS CARGOS:")

comparabilidade_cargos = (df_perfis
    .select("edicao","cargo_atual")
    .distinct()
    .groupBy("cargo_atual")
    .agg(F.countDistinct("edicao").alias("qtd_edicoes"))
    .orderBy("qtd_edicoes","cargo_atual")
)
comparabilidade_cargos.show(100,truncate=False)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Júnior, Pleno e Sênior são comparáveis nas três edições.
A nomenclatura de Engenharia e Arquitetura de Dados mudou ao longo das pesquisas, exigindo harmonização para análise histórica.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------



# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# INVESTIGAÇÃO DA FAIXA SALARIAL INCONSISTENTE
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("5. INVESTIGAÇÃO DA FAIXA SALARIAL INCONSISTENTE")
print("=" * 100)

(df_perfis
    .filter(F.col("valor")== "de R$ 101/mês a R$ 2.000/mês")
    .select("edicao","cargo_atual","nivel","valor","contagem","total_respondentes","pct_na_dimensao")
    .show(truncate=False)
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A faixa inconsistente ocorre uma única vez: em 2023-2024, para Engenharia de Dados, nível Júnior.
O registro representa 1 de 125 respondentes (0,8%) e foi removido antes das análises.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------



# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# PREPARAÇÃO DA BASE ANALÍTICA
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("6. PREPARAÇÃO DA BASE ANALÍTICA")
print("=" * 100)

# REMOVER A FAIXA SALARIAL INCONSISTENTE ---
df_perfis_tratado = (df_perfis
    .filter((F.col("valor")!= "de R$ 101/mês a R$ 2.000/mês") | F.col("valor").isNull())
)

# HARMONIZAR ENGENHARIA E ARQUITETURA DE DADOS ---
df_perfis_tratado = (df_perfis_tratado.withColumn("cargo_harmonizado",
        F.when(
            F.col("cargo_atual").isin(
                "Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect",
                "Engenheiro de Dados/Data Engineer/Data Architect",
                "Arquiteto de Dados/Data Architect"
            ),
            "Engenharia e Arquitetura de Dados")
        .otherwise(F.col("cargo_atual"))
    )
)

# CONSOLIDAR CONTAGENS APÓS A HARMONIZAÇÃO ---
df_perfis_consolidado = (df_perfis_tratado
    .groupBy("edicao","cargo_harmonizado","nivel","valor")
    .agg(F.sum("contagem").alias("contagem"))
)

# RECALCULAR O DENOMINADOR ---
janela_perfil = (Window.partitionBy("edicao","cargo_harmonizado","nivel"))

df_perfis_consolidado = (df_perfis_consolidado
    .withColumn("total_respondentes",F.sum("contagem").over(janela_perfil))
    .withColumn("pct_na_dimensao",F.round((F.col("contagem")/ F.col("total_respondentes")) * 100,2))
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# VALIDAÇÃO DA BASE TRATADA
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("7. VALIDAÇÃO DA BASE TRATADA")
print("=" * 100)

validacao_base_tratada = (df_perfis_consolidado
    .groupBy("edicao","cargo_harmonizado","nivel")
    .agg(F.sum("contagem").alias("total_respondentes"),
        F.round(F.sum("pct_na_dimensao"),2).alias("soma_percentual"))
    .orderBy("edicao", "cargo_harmonizado","nivel")
)

# GRUPOS FORA DO INTERVALO ESPERADO ---
print("\nGRUPOS COM PERCENTUAL FORA DO INTERVALO 99,9% A 100,1%:")

(validacao_base_tratada
    .filter((F.col("soma_percentual") < 99.9) | (F.col("soma_percentual") > 100.1))
    .show(100,truncate=False)
)

# VALIDAR HARMONIZAÇÃO DE ENGENHARIA E ARQUITETURA ---
print("\nENGENHARIA E ARQUITETURA DE DADOS APÓS HARMONIZAÇÃO:")

(validacao_base_tratada
    .filter(F.col("cargo_harmonizado")== "Engenharia e Arquitetura de Dados")
    .orderBy("edicao","nivel")
    .show(100,truncate=False)
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Após o tratamento e a harmonização, todos os grupos apresentam soma percentual entre 99,9% e 100,1%.
A base tratada está consistente para as análises de remuneração.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# ORDENAÇÃO DAS FAIXAS SALARIAIS
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("8. ORDENAÇÃO DAS FAIXAS SALARIAIS")
print("=" * 100)

df_perfis_ordenado = (df_perfis_consolidado
    .withColumn("ordem_faixa_salarial",F.when(F.col("valor") == "Menos de R$ 1.000/mês",1)
        .when(F.col("valor") == "de R$ 1.001/mês a R$ 2.000/mês",2)
        .when(F.col("valor") == "de R$ 2.001/mês a R$ 3.000/mês",3)
        .when(F.col("valor") == "de R$ 3.001/mês a R$ 4.000/mês",4)
        .when(F.col("valor") == "de R$ 4.001/mês a R$ 6.000/mês",5)
        .when(F.col("valor") == "de R$ 6.001/mês a R$ 8.000/mês",6)
        .when(F.col("valor") == "de R$ 8.001/mês a R$ 12.000/mês",7)
        .when(F.col("valor") == "de R$ 12.001/mês a R$ 16.000/mês",8)
        .when(F.col("valor") == "de R$ 16.001/mês a R$ 20.000/mês",9)
        .when(F.col("valor") == "de R$ 20.001/mês a R$ 25.000/mês",10)
        .when(F.col("valor") == "de R$ 25.001/mês a R$ 30.000/mês",11)
        .when(F.col("valor") == "de R$ 30.001/mês a R$ 40.000/mês",12)
        .when(F.col("valor") == "Acima de R$ 40.001/mês",13))
)

# VALIDAR ORDEM ---
(df_perfis_ordenado
    .select("valor","ordem_faixa_salarial")
    .distinct()
    .orderBy("ordem_faixa_salarial")
    .show(100,truncate=False)
)

# VALIDAR SE ALGUMA FAIXA FICOU SEM ORDEM ---
print("\nFAIXAS SEM ORDEM DEFINIDA:")

(df_perfis_ordenado
    .filter(F.col("ordem_faixa_salarial").isNull())
    .select("valor")
    .distinct()
    .show(100,truncate=False)
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
As 13 faixas salariais válidas foram ordenadas corretamente e nenhuma ficou sem classificação.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
